# NetraVision - Export UrbanVision Seg ke TFLite (ringan)

Notebook ini **bukan training**. Hanya mengunduh bobot `.pt` yang sudah jadi lalu mengekspornya ke `.tflite` yang lebih kecil.

**Penting (kompatibilitas plugin Flutter `flutter_vision`):**
- Plugin hanya menerima input **float32**.
- Maka **JANGAN pakai full int8 (`int8=True`)** -> input jadi int8 -> crash.
- Gunakan salah satu:
  1. `quantize="w8a32"` -> dynamic-range INT8 (int8 weights, float32 activations) ~60 MB, **disarankan**.
  2. `half=True` (float16) -> ~124 MB, akurasi paling aman.
- Selalu `nms=False` agar output mentah (2 output) sesuai parser segmentasi plugin.

Cara pakai: Runtime > Change runtime type > T4 GPU (opsional), lalu jalankan tiap sel dari atas ke bawah.

In [ ]:
!pip install -q ultralytics

In [ ]:
from huggingface_hub import hf_hub_download

pt_path = hf_hub_download(
    repo_id="khanhromvn/urbanvision-seg-v1",
    filename="urbanvision-seg.pt",
)
print("Downloaded:", pt_path)

## Export

Default di bawah memakai **w8a32** (paling ringan yang tetap kompatibel & tanpa gambar kalibrasi).
Kalau ingin akurasi paling aman, ganti `quantize="w8a32"` menjadi `half=True`.

In [ ]:
from ultralytics import YOLO

# Setelah download .pt, ganti ke jalur yang disediakan. Default: T4 GPU (boleh diubah ke "cpu").
assert 'pt_path' in globals(), 'Pilih file .pt terlebih dahulu pada sel sebelumnya'

model = YOLO(pt_path)
out = model.export(
    format="tflite",
    quantize="w8a32",   # int8 weights + float32 activations (kompatibel flutter_vision)
    imgsz=416,           # 640 -> 416: ~2.4x lebih cepat & ringan di HP kelas menengah (Redmi Note 9)
    nms=False,           # WAJIB: biarkan output mentah (2 output) untuk segmentasi
)
print("Exported:", out)

Kalau sel di atas error karena `quantize` tidak dikenali versi Ultralytics Anda, pakai alternatif float16:

In [ ]:
# ALTERNATIF (jalankan hanya bila perlu):
# out = model.export(format="tflite", half=True, imgsz=640, nms=False)
# print("Exported (float16):", out)

## Verifikasi output

Cek ukuran file, jumlah output (harus **2** untuk segmentasi), dan tipe input (harus **float32**).

In [ ]:
import glob, os
import tensorflow as tf

files = sorted(glob.glob('*_saved_model/*.tflite'))
if not files:
    raise FileNotFoundError('Tidak ada file .tflite. Pastikan export berhasil.')

for f in files:
    print(f'{f}  ->  {os.path.getsize(f)/1e6:.1f} MB')

chosen = out if isinstance(out, str) and out.endswith('.tflite') else files[-1]
print('Model dipilih:', chosen)

it = tf.lite.Interpreter(model_path=chosen)
it.allocate_tensors()
inp = it.get_input_details()[0]
outs = it.get_output_details()
print('INPUT :', inp['shape'], inp['dtype'])
print('JUMLAH OUTPUT:', len(outs), '(harus 2 untuk segmentasi)')
for d in outs:
    print('  OUTPUT:', d['shape'], d['dtype'])
if inp['dtype'] != tf.float32:
    print('\n!!! PERINGATAN: input bukan float32 -> TIDAK kompatibel dengan flutter_vision !!!')

## Unduh hasil

File `.tflite` hasil akan diunduh ke komputer Anda. Simpan sebagai `assets/models/outdoor/netravision_out.tflite` (ganti file lama yang 248 MB).

In [ ]:
from google.colab import files
files.download(chosen)